# Code to prepare transformer training for whole Africa

In [19]:
import os
import sys
sys.path.insert(1, "/home/users/mendrika/SSA/SA/module")
import torch
import snflics
import numpy as np      
from netCDF4 import Dataset  
from scipy.ndimage import label, zoom
from datetime import datetime, timedelta

# Useful functions

In [20]:
def prepare_core(file):

    if not os.path.exists(file):
        raise FileNotFoundError(f"The file '{file}' does not exist.")
    try:
        # Open the NetCDF file using a context manager to ensure proper file closure
        with Dataset(file, "r") as data:
            cores = data.variables["cores"][0, :, :]
    except OSError as e:
        raise OSError(f"Error opening NetCDF file: {file}. {e}")
    
    return cores

In [21]:
def update_hour(date_dict, hours_to_add):
    """
    Add hours to a datetime dictionary and return the updated dict and a generated file path.

    Args:
        date_dict (dict): Keys: 'year', 'month', 'day', 'hour', 'minute' as strings, e.g. "01", "23"
        hours_to_add (int): Number of hours to add.

    Returns:
        tuple:
            - dict: Updated datetime dictionary with all fields as zero-padded strings.
            - str: File path in the format YYYY/MM/YYYYMMDDHHMM.nc
    """
    # Parse the original time
    time_obj = datetime(
        int(date_dict["year"]),
        int(date_dict["month"]),
        int(date_dict["day"]),
        int(date_dict["hour"]),
        int(date_dict["minute"])
    )

    # Add hours
    updated = time_obj + timedelta(hours=hours_to_add)

    # Format updated dictionary
    new_date_dict = {
        "year": f"{updated.year:04d}",
        "month": f"{updated.month:02d}",
        "day": f"{updated.day:02d}",
        "hour": f"{updated.hour:02d}",
        "minute": f"{updated.minute:02d}"
    }

    # Generate file path
    file_path = f"{new_date_dict['year']}/{new_date_dict['month']}/{new_date_dict['year']}{new_date_dict['month']}{new_date_dict['day']}{new_date_dict['hour']}{new_date_dict['minute']}.nc"

    return new_date_dict, file_path

In [22]:
def haversine_distance(lat1, lon1, lat2, lon2):
        """
        Compute Haversine distance between two points or arrays of points.
        Inputs are in degrees. Output is in kilometers.
        
        Supports both scalar and array inputs (NumPy).
        """
        R = 6371.0  # Earth radius in kilometers

        # Convert degrees to radians
        lat1_rad = np.radians(lat1)
        lon1_rad = np.radians(lon1)
        lat2_rad = np.radians(lat2)
        lon2_rad = np.radians(lon2)

        dlat = lat2_rad - lat1_rad
        dlon = lon2_rad - lon1_rad

        a = np.sin(dlat / 2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2)**2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

        return R * c

In [23]:
def create_storm_database(data_t0, lats, lons):
    """
    Identify storm cores within the context domain and extract features for each core.

    Parameters:
        x0_lat (np.ndarray): Latitude coordinates of core points.
        x0_lon (np.ndarray): Longitude coordinates of core points.
        data_t0 (netCDF4.Dataset): Dataset at t0 containing 'cores' variable.
        lats, lons (np.ndarray): 2D arrays of grid latitudes and longitudes.

    Returns:
        dict: Dictionary of storm features indexed by core label.
    """
    cores_t0 = data_t0["cores"][0, :, :]
    x0_lat = data_t0["max_lat"][:]
    x0_lon = data_t0["max_lon"][:]

    labeled_array, _ = label(cores_t0 != 0)
    core_labels = np.unique(labeled_array[labeled_array != 0])

    dict_storm_size = {
        core_label: np.sum(labeled_array == core_label) * 9
        for core_label in core_labels
    }

    dict_storm_intensity = {
        core_label: np.mean(cores_t0[labeled_array == core_label])
        for core_label in core_labels
    }

    storm_database = {}
    for lat, lon in zip(x0_lat, x0_lon):
        try:
            x0_y, x0_x = snflics.to_yx(lat, lon, lats, lons)
        except IndexError:
            continue                   # Skip points that fall outside the coordinate grid

        lab = labeled_array[x0_y, x0_x]

        if lab == 0 or lab in storm_database:
            continue

        storm_database[int(lab)] = {
            "lat": lat,
            "lon": lon,
            "wp": dict_storm_intensity[lab],
            "size": dict_storm_size[lab],
            "mask": 1
        }
    return storm_database

In [24]:
def load_wavelet_dataset(year, month, day, hour, minute):
    path_core = f'/gws/nopw/j04/cocoon/SSA_domain/ch9_wavelet/{year}/{month}'
    file = f'{path_core}/{year}{month}{day}{hour}{minute}.nc'
    return Dataset(file, mode='r')

In [25]:
def resize_core(original_core, target_shape_y, target_shape_x):
    """
    Resize a 2D array using bilinear interpolation to the target shape.

    Parameters:
    - original_core: 2D np.ndarray
    - target_shape_y: int, desired number of rows
    - target_shape_x: int, desired number of columns

    Returns:
    - resized array of shape (target_shape_y, target_shape_x)
    """
    assert original_core.ndim == 2,                     "Input must be a 2D array"
    assert target_shape_y > 0 and target_shape_x > 0,   "Target dimensions must be positive"

    zoom_factors = (
        target_shape_y / original_core.shape[0],
        target_shape_x / original_core.shape[1],
    )

    return zoom(original_core, zoom=zoom_factors, order=1)

In [26]:
def generate_fictional_storm(
        context_lat_min, context_lat_max,
        context_lon_min, context_lon_max,
        min_km_buffer=500,
        max_deg_buffer=4.5):
    """
    Generate a fictional storm at least `min_km_buffer` km from the context domain,
    but not farther than `max_deg_buffer` degrees away.

    Parameters:
        context_lat_min, context_lat_max: float
            Latitude bounds of the context domain.
        context_lon_min, context_lon_max: float
            Longitude bounds of the context domain.
        min_km_buffer: float
            Minimum required distance from context domain edge.
        max_deg_buffer: float
            Maximum distance (in degrees) allowed from context edge for sampling.

    Returns:
        tuple: (0, dict with lat, lon, wp, size, distance, mask)
    """

    # Sampling range: context domain ± max_deg_buffer
    lat_range = (context_lat_min - max_deg_buffer, context_lat_max + max_deg_buffer)
    lon_range = (context_lon_min - max_deg_buffer, context_lon_max + max_deg_buffer)

    while True:
        lat = np.random.uniform(*lat_range)
        lon = np.random.uniform(*lon_range)

        # Reject if inside context domain
        if context_lat_min <= lat <= context_lat_max and context_lon_min <= lon <= context_lon_max:
            continue

        # Compute distance to each context edge
        d_north = haversine_distance(lat, lon, context_lat_max, lon)
        d_south = haversine_distance(lat, lon, context_lat_min, lon)
        d_east  = haversine_distance(lat, lon, lat, context_lon_max)
        d_west  = haversine_distance(lat, lon, lat, context_lon_min)

        min_edge_dist = min(d_north, d_south, d_east, d_west)

        if min_edge_dist < min_km_buffer:
            continue

        # Accept
        return ('artificial', {
            'lat': lat,
            'lon': lon,
            'wp': 0.0,
            'size': 0,
            'mask': 0
        })

In [27]:
def pad_observed_storms(storm_db, nb_x0, context_lat_min, context_lat_max, context_lon_min, context_lon_max):
    # Convert dict to list of (key, value) tuples
    storm_list = list(storm_db.items())

    if len(storm_list) >= nb_x0:
        # taking the nb_x0 strongest cores if there are more than the max number of cores allowed by the model
        sorted_db = sorted(storm_list, key=lambda item: item[1]['wp'], reverse=True)
        return sorted_db[:nb_x0]
    else:
        # apply padding when there are less cores observed at time t0
        needed = nb_x0 - len(storm_list)
        storm_list.extend([
            generate_fictional_storm(
                context_lat_min=context_lat_min, 
                context_lat_max=context_lat_max, 
                context_lon_min=context_lon_min,
                context_lon_max=context_lon_max
            ) 
            for _ in range(needed)
        ])
        return storm_list

In [28]:
geodata = np.load("/home/users/mendrika/EPS-Impact-Case-AI-Nowcasting/data/geodata/lat_lon_2268_2080.npz")
lons = geodata["lon"][:]
lats = geodata["lat"][:]

In [29]:
# Target domain
TARGET_DOMAIN_LAT_MIN = -40
TARGET_DOMAIN_LAT_MAX = 40

TARGET_DOMAIN_LON_MIN = -25
TARGET_DOMAIN_LON_MAX = 60

# Context domain
CONTEXT_DOMAIN_LAT_MIN = -46
CONTEXT_DOMAIN_LAT_MAX = 46

CONTEXT_DOMAIN_LON_MIN = -31
CONTEXT_DOMAIN_LON_MAX = 66

In [30]:
example_time = {
    'year': '2021',
    'month': '07',
    'day': '16',
    'hour': '16',
    'minute': '00' 
}

In [31]:
example_data = load_wavelet_dataset('2021', '07', '16', '16', '00')
example_core = example_data['cores'][0, :, :]
db_storm = create_storm_database(example_data, lats, lons)
X0 = pad_observed_storms(db_storm, 60, context_lat_min=CONTEXT_DOMAIN_LAT_MIN, context_lat_max=CONTEXT_DOMAIN_LAT_MAX, context_lon_min=CONTEXT_DOMAIN_LON_MIN, context_lon_max=CONTEXT_DOMAIN_LON_MAX)

In [32]:
X0

[(1,
  {'lat': np.float32(-2.5340195),
   'lon': np.float32(21.511192),
   'wp': np.float64(118.70886075949367),
   'size': np.int64(711),
   'mask': 1}),
 (2,
  {'lat': np.float32(-2.1231987),
   'lon': np.float32(22.445856),
   'wp': np.float64(100.45454545454545),
   'size': np.int64(2178),
   'mask': 1}),
 (3,
  {'lat': np.float32(-1.7608147),
   'lon': np.float32(20.83489),
   'wp': np.float64(179.34451219512195),
   'size': np.int64(2952),
   'mask': 1}),
 (4,
  {'lat': np.float32(-1.3192879),
   'lon': np.float32(20.112022),
   'wp': np.float64(114.36363636363636),
   'size': np.int64(198),
   'mask': 1}),
 (5,
  {'lat': np.float32(-0.98966086),
   'lon': np.float32(20.317915),
   'wp': np.float64(146.6875),
   'size': np.int64(288),
   'mask': 1}),
 (6,
  {'lat': np.float32(-0.88958263),
   'lon': np.float32(28.05636),
   'wp': np.float64(93.375),
   'size': np.int64(144),
   'mask': 1}),
 (7,
  {'lat': np.float32(4.6969337),
   'lon': np.float32(22.920961),
   'wp': np.float64

In [33]:
def transform_to_array(time_obs, data):
    """
    Transform list of (id, dict) into numpy array.

    Output shape: (N, 5), with columns: lat, lon, wp, size, mask
    """

    year = int(time_obs['year'])
    month = int(time_obs['month'])
    day = int(time_obs['day'])
    hour = int(time_obs['hour'])
    minute = int(time_obs['hour'])
    result = []
    
    for _, entry in data:
        lat = float(entry['lat'])
        lon = float(entry['lon'])
        wp = float(entry['wp'])
        size = int(entry['size'])
        mask = int(entry['mask'])
        result.append([year, month, day, hour, minute, lat, lon, wp, size, mask])
    
    return np.array(result)


In [34]:
transform_to_array(example_time, X0).shape

(60, 10)

In [35]:
YEAR = "2021"

In [ ]:
TARGET_SHAPE_Y, TARGET_SHAPE_X = 1028, 1028

# Data and output paths
DATA_PATH = "/gws/nopw/j04/cocoon/SSA_domain/ch9_wavelet/"

# Months of interest
all_files = [file for file in snflics.all_files_in(DATA_PATH) if snflics.get_time(file)["year"] == YEAR]
all_files.sort()

# Number of storms to consider (after analysing the whole dataset)
NB_X0 = 2

for file_t0 in all_files[:10]:
    
    time_t0 = snflics.get_time(file_t0)

    # file name for lead time from 0 to 6 hours
    files = [DATA_PATH + update_hour(time_t0, h)[1] for h in range(7)]     

    INPUT_LT0 = f"/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Africa/inputs_t0/input-{time_t0['year']}{time_t0['month']}{time_t0['day']}_{time_t0['hour']}{time_t0['minute']}.pt"
    OUTPUT_PATHS = {
        f"LT{i}": f"/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Africa/targets_t{i}/target-{time_t0['year']}{time_t0['month']}{time_t0['day']}_{time_t0['hour']}{time_t0['minute']}.pt"
        for i in range(7)
    }

    # If all forecast files exist
    if all(os.path.exists(f) for f in files):
        try:
            core_series = [prepare_core(f) for f in files]
        except OSError:
            continue

        with Dataset(file_t0, "r") as data_t0:
            x0_lat = data_t0["max_lat"][:]
            x0_lon = data_t0["max_lon"][:]

            if x0_lat.size != 0:
                # database of all identified storms
                storm_database = create_storm_database(data_t0, lats, lons)

                # taking a certain number of closest storms
                X0_features = pad_observed_storms(
                    storm_database, NB_X0,
                    context_lat_min=CONTEXT_DOMAIN_LAT_MIN, context_lat_max=CONTEXT_DOMAIN_LAT_MAX,
                    context_lon_min=CONTEXT_DOMAIN_LON_MIN, context_lon_max=CONTEXT_DOMAIN_LON_MAX
                )

                input_features = transform_to_array(time_t0, X0_features)
                input_tensor = torch.tensor(input_features, dtype=torch.float32)
                
                # Save input directly as torch tensor
                torch.save(input_tensor, INPUT_LT0)

                # Process targets for each lead time
                for i, core in enumerate(core_series):
                    resized_core = resize_core(core, TARGET_SHAPE_Y, TARGET_SHAPE_X)
                    Cb = (resized_core != 0).astype(np.uint8)                               # no need to flatten for CNN
                    target_tensor = torch.tensor(Cb, dtype=torch.uint8)
                    output_file_path = OUTPUT_PATHS[f"LT{i}"]
                    torch.save(target_tensor, output_file_path)

0 2006
1 2329
2 2316
3 2015
4 2291
5 2353
6 1807
0 1823
1 2408
2 2180
3 2186
4 2295
5 2317
6 1550
0 1956
1 2428
2 2157
3 2274
4 2356
5 2221
6 1287
0 2198
1 2397
2 1993
3 2307
4 2395
5 2072
6 1149
0 2329
1 2316
2 2015
3 2291
4 2353
5 1807
6 1045
0 2408
1 2180
2 2186
3 2295
4 2317
5 1550
6 1072
0 2428
1 2157
2 2274
3 2356
4 2221
5 1287
6 1108
0 2397
1 1993
2 2307
3 2395
4 2072
5 1149
6 1082
0 2316
1 2015
2 2291
3 2353
4 1807
5 1045
6 1042
0 2180
1 2186
2 2295
3 2317
4 1550
5 1072
6 889
